# Simurgh — Phase 0 Demo: Naive RAG (lexical BM25)

The simplest rung of the RAG ladder: lexical **BM25** retrieval over a Persian corpus (SQLite FTS5) plus an OpenAI-compatible LLM for generation. No personalization, no RL.

Steps: load config → ingest a Persian document → retrieve (no API key needed) → generate a grounded answer.

> Run this with the project's virtual-env kernel:
> `uv run --extra dev jupyter lab`  (or select the `.venv` kernel).

In [ ]:
from pathlib import Path

from rag.pipeline import NaiveRAG, load_config

# Resolve the project root so paths work whether the kernel runs from the repo root or notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

config = load_config(ROOT / "configs" / "phase0_naive.yaml")
# Make the index path absolute so it does not depend on the kernel's working directory.
config["knowledge_base"]["db_path"] = str(ROOT / "data" / "index" / "demo.db")
config

## 1) Build the system and ingest the knowledge base

In [ ]:
rag = NaiveRAG(config)

# Rebuild the index on every run so re-executing the notebook does not duplicate chunks.
rag.kb.clear()
n_chunks = rag.ingest([ROOT / "data" / "raw" / "farsi-9th-grade.md"])
print(f"Indexed {n_chunks} chunks | total in store: {rag.kb.count()}")

## 2) Retrieval without an LLM (no API key required)

This step uses only BM25 keyword search — no API key or model server needed. It is the right way to check the retriever in isolation ("evaluate retrieval before generation").

In [ ]:
def show_hits(hits):
    if not hits:
        print("No results.")
        return
    for rank, h in enumerate(hits, start=1):
        print(f"#{rank}  score={h.score:.3f}  [{h.source}#{h.chunk_index}]")
        print(f"     {h.raw_text[:120]} ...")
        print()


# The corpus is the 9th-grade Persian literature textbook, so queries must be Persian to match.
# (Query means: "What is the simile literary device in poetry?")
query = "آرایه تشبیه در شعر چیست؟"
show_hits(rag.retrieve(query))

## 3) Grounded generation (requires an LLM endpoint)

This calls the model configured in the YAML. Start a local server (LMStudio at `http://localhost:1234/v1`, or llama.cpp), or point `OPENAI_BASE_URL` / `OPENAI_API_KEY` at OpenAI or Google AI Studio in a `.env` file (copy from `.env.example`).

In [ ]:
# (Question means: "What is the difference between the simile and personification literary devices?")
question = "تفاوت آرایه تشبیه و جان‌بخشی چیست؟"
try:
    result = rag.ask(question)
    print("Answer:\n")
    print(result.answer)
    print("\n" + "-" * 60)
    print("Sources used in retrieval:\n")
    show_hits(result.hits)
except Exception as exc:
    print("LLM call failed. Is an OpenAI-compatible server running?")
    print(
        "Hint: run LMStudio at http://localhost:1234/v1, or set OPENAI_BASE_URL / OPENAI_API_KEY in .env."
    )
    print(f"\nError detail: {exc}")

## 4) Compare prompt variants (EN vs FA)

Same question, both prompts, side by side. English vs Persian instructions can differ for small or local models; a rigorous comparison waits for the eval harness. Needs an LLM endpoint.

In [ ]:
# Same question through both prompt variants (needs an LLM endpoint).
for variant in ("en", "fa"):
    print(f"=== prompt_variant = {variant} ===")
    try:
        print(rag.ask(question, variant=variant).answer)
    except Exception as exc:
        print(f"LLM call failed: {exc}")
    print()

## Switching providers

The same `OpenAICompatClient` works with any OpenAI-compatible endpoint — only `OPENAI_BASE_URL` / `OPENAI_API_KEY` (in `.env`) and the `model` (in the YAML) change:

| Provider | OPENAI_BASE_URL | model (example) |
|---|---|---|
| OpenAI | https://api.openai.com/v1 | gpt-4o-mini |
| Google AI Studio | https://generativelanguage.googleapis.com/v1beta/openai/ | gemini-2.0-flash |
| LMStudio | http://localhost:1234/v1 | (local model id) |
| llama.cpp | http://localhost:8080/v1 | (local model id) |

In [ ]:
rag.close()